# Lezione 06 — Testare i sistemi di ML

**Corso:** MLOps e Monitoraggio con Python · **Blocco 2:** Versionare e misurare
**Tempo stimato:** 100 minuti · **Capitolo del manuale:** `manual/06_probar_sistemas_ml_IT.md`

## Cosa vedrai in questa lezione

- **Test unitari** con `pytest` per funzioni REALI del progetto (pulizia, trasformazione, valutazione), scritte in un file reale ed eseguite con `subprocess.run(["pytest", ...])`.
- **Test dei dati** con `pandera`: uno schema con tipi, intervalli, valori ammessi e proporzione di nulli, con casi di dati corrotti APPOSTA che devono essere rifiutati.
- **Test del modello** con una **soglia di regressione**: il modello nuovo non può essere peggiore del precedente di più di una tolleranza, con un caso che passa e uno che fallisce apposta (un modello peggiore, costruito senza bilanciamento).
- I **limiti** di una soglia fissa: non rileva una deriva lenta né un problema localizzato in un sottogruppo.
- I tre tipi di test, integrati in un'unica suite eseguibile con un comando.

Tutto avviene dentro una cartella di lavoro effimera (`proyecto_churn/`), creata all'inizio ed **eliminata alla fine** di questo stesso notebook.

## Prerequisiti

- `pandas`, `scikit-learn`, `pytest` e `pandera` (la cella seguente installa solo ciò che manca, anche in Google Colab).
- La cartella `datos/` del corso, con `churn.csv` all'interno, e aver letto `datos/LEEME.md`.
- Dalla lezione 5: la `Pipeline` completa (pulizia + `ColumnTransformer` + modello) e la configurazione 3 (regressione logistica, `class_weight="balanced"`), che riusiamo qui.

> **Come usare questo notebook:** esegui le celle in ordine, dall'alto in basso, una sola volta. Richiede meno di un minuto in totale.

---
## Installazione

Installa solo ciò che manca.

In [ ]:
import subprocess
import sys


def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)


asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("scikit-learn", "sklearn")
asegurar("pytest", "pytest")
asegurar("pandera", "pandera")
print("Librerie pronte.")

---
## Cella 0 — Setup

Importiamo le librerie, definiamo `cargar_datos`, `SEMILLA = 42` e `ejecutar_pytest` (una funzione ausiliaria che esegue pytest e riassume il risultato SENZA i tempi di esecuzione, che cambiano a ogni run).

In [ ]:
import os
import re
import shutil
import subprocess
import sys
import json

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import pandera.pandas as pa
from pandera.pandas import Column, DataFrameSchema, Check

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/18-mlops-monitoraggio/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

SEMILLA = 42
COLUMNAS_CATEGORICAS = ["plan", "contrato", "metodo_pago"]
COLUMNAS_NUMERICAS = ["antiguedad_meses", "uso_mensual_horas", "tickets_soporte", "factura_mensual"]


def ejecutar_pytest(argumentos):
    """Esegue pytest con subprocess.run e restituisce un riepilogo SENZA i tempi di esecuzione (cambiano a ogni esecuzione, non si possono confrontare tra le classi)."""
    resultado = subprocess.run([sys.executable, "-m", "pytest", "-q"] + argumentos, capture_output=True, text=True)
    salida = resultado.stdout + resultado.stderr
    m_passed = re.search(r"(\d+) passed", salida)
    m_failed = re.search(r"(\d+) failed", salida)
    m_error = re.search(r"(\d+) error", salida)
    return {
        "returncode": resultado.returncode,
        "passed": int(m_passed.group(1)) if m_passed else 0,
        "failed": int(m_failed.group(1)) if m_failed else 0,
        "error": int(m_error.group(1)) if m_error else 0,
    }


print("Librerie e funzioni ausiliarie pronte.")

---
## Cartella di lavoro effimera

Come nelle lezioni precedenti: creiamo `proyecto_churn/` da zero.

In [ ]:
CARPETA = "proyecto_churn"

# Puliamo e ricreiamo la cartella perché il notebook sia ripetibile da zero
if os.path.exists(CARPETA):
    shutil.rmtree(CARPETA)
os.makedirs(CARPETA)
print("Cartella di lavoro creata:", os.path.abspath(CARPETA))

---
## 1. Il codice reale del progetto: `proyecto.py`

Prima di scrivere qualsiasi test serve del codice da testare. Scriviamo un file reale, `proyecto_churn/proyecto.py`, con **una funzione di pulizia** (`limpiar_tickets_soporte`), **una funzione di trasformazione** (`categorizar_antiguedad`, lo stesso raggruppamento della lezione 5), **una funzione di valutazione** (`calcular_metricas`), lo **schema dei dati** con `pandera` (`ESQUEMA_CHURN`) e la **Pipeline del modello** (`construir_pipeline`, identica alla lezione 5, con un parametro `balanceado` per poter costruire apposta una versione peggiore nella sezione 4).

In [ ]:
PROYECTO_PY = """
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from pandera.pandas import Column, DataFrameSchema, Check

SEMILLA = 42
COLUMNAS_CATEGORICAS = ["plan", "contrato", "metodo_pago"]
COLUMNAS_NUMERICAS = ["antiguedad_meses", "uso_mensual_horas", "tickets_soporte", "factura_mensual"]


def cargar_datos(nombre):
    '''Cerca un file di dati in datos/, ../datos/ e /content/datos/.'''
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/18-mlops-monitoraggio/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError("Non trovo " + nombre)


# ----- Funzione di PULIZIA -----
def limpiar_tickets_soporte(serie):
    '''Un numero di ticket di supporto non può mai essere negativo: qualsiasi valore negativo (un errore di inserimento) viene ricondotto a 0.'''
    return serie.clip(lower=0)


# ----- Funzione di TRASFORMAZIONE -----
def categorizar_antiguedad(meses):
    '''Raggruppa antiguedad_meses in 'nuevo' (<12), 'medio' (12-36) e 'antiguo' (>36).'''
    if meses < 12:
        return "nuevo"
    elif meses <= 36:
        return "medio"
    return "antiguo"


# ----- Funzione di VALUTAZIONE -----
def calcular_metricas(y_true, y_pred, y_proba):
    '''Accuratezza, F1 e AUC arrotondate a 4 decimali, come nella lezione 5.'''
    return {
        "exactitud": round(accuracy_score(y_true, y_pred), 4),
        "f1": round(f1_score(y_true, y_pred), 4),
        "auc": round(roc_auc_score(y_true, y_proba), 4),
    }


# ----- Schema dei DATI (pandera) -----
ESQUEMA_CHURN = DataFrameSchema(
    {
        "id_cliente": Column(str, unique=True, nullable=False, checks=Check.str_startswith("C")),
        "antiguedad_meses": Column(int, Check.in_range(0, 120), nullable=False),
        "plan": Column(str, Check.isin(["basico", "estandar", "premium"]), nullable=False),
        "contrato": Column(str, Check.isin(["mensual", "anual"]), nullable=False),
        "uso_mensual_horas": Column(float, Check.in_range(0, 200), nullable=False),
        "tickets_soporte": Column(int, Check.in_range(0, 20), nullable=False),
        "metodo_pago": Column(str, Check.isin(["tarjeta", "domiciliacion", "transferencia"]), nullable=False),
        "factura_mensual": Column(float, Check.ge(0), nullable=False),
        "abandono": Column(int, Check.isin([0, 1]), nullable=False),
    },
    # Controllo a livello dell'intero DataFrame: nessuna colonna può superare il 5% di nulli
    checks=Check(lambda d: d.isna().mean().max() <= 0.05, element_wise=False),
)


# ----- Pipeline del modello (stessa ricetta della lezione 5) -----
def construir_pipeline(balanceado=True):
    preprocesado = ColumnTransformer([
        ("categoricas", Pipeline([
            ("imputar", SimpleImputer(strategy="most_frequent")),
            ("codificar", OneHotEncoder(handle_unknown="ignore")),
        ]), COLUMNAS_CATEGORICAS),
        ("numericas", Pipeline([
            ("imputar", SimpleImputer(strategy="median")),
            ("escalar", StandardScaler()),
        ]), COLUMNAS_NUMERICAS),
    ])
    peso = "balanced" if balanceado else None
    return Pipeline([
        ("preprocesado", preprocesado),
        ("modelo", LogisticRegression(class_weight=peso, random_state=SEMILLA, max_iter=1000)),
    ])


def preparar_datos():
    '''Carica churn.csv e separa addestramento/test, come nella lezione 5.'''
    ruta = cargar_datos("churn.csv")
    df = pd.read_csv(ruta)
    X = df[COLUMNAS_CATEGORICAS + COLUMNAS_NUMERICAS]
    y = df["abandono"]
    return train_test_split(X, y, test_size=0.25, random_state=SEMILLA, stratify=y)
"""

In [ ]:
ruta_proyecto = os.path.join(CARPETA, "proyecto.py")
with open(ruta_proyecto, "w", encoding="utf-8") as f:
    f.write(PROYECTO_PY)
print("file del progetto creato in:", ruta_proyecto)

---
## 2. Test unitari con `pytest`

Scriviamo `proyecto_churn/test_funciones.py` con test REALI per le tre funzioni della sezione 1: casi normali e casi limite (i valori `12` e `36` di `categorizar_antiguedad`, dove è facile programmare `<` invece di `<=`, come visto nella lezione 5). Li eseguiamo per davvero con `subprocess.run(["pytest", ...])`, esattamente come si eseguirebbero da un terminale.

In [ ]:
TEST_FUNCIONES_PY = """
import pandas as pd
from proyecto import limpiar_tickets_soporte, categorizar_antiguedad, calcular_metricas


def test_limpiar_tickets_soporte_recorta_negativos():
    '''Un valore negativo (impossibile nella realtà) deve diventare 0.'''
    entrada = pd.Series([2, -1, 0, 5, -3])
    resultado = limpiar_tickets_soporte(entrada)
    assert resultado.tolist() == [2, 0, 0, 5, 0]


def test_limpiar_tickets_soporte_no_toca_valores_validos():
    entrada = pd.Series([1, 2, 3])
    assert limpiar_tickets_soporte(entrada).tolist() == [1, 2, 3]


def test_categorizar_antiguedad_en_los_limites():
    '''I limiti esatti (12 e 36) sono il caso più facile da programmare male (< invece di <=).'''
    assert categorizar_antiguedad(11) == "nuevo"
    assert categorizar_antiguedad(12) == "medio"
    assert categorizar_antiguedad(36) == "medio"
    assert categorizar_antiguedad(37) == "antiguo"


def test_calcular_metricas_con_valores_conocidos():
    '''Metriche calcolate a mano con sklearn su un esempio piccolo (6 righe).'''
    y_true = [0, 0, 0, 1, 1, 1]
    y_pred = [0, 0, 1, 1, 0, 1]
    y_proba = [0.1, 0.2, 0.6, 0.8, 0.3, 0.9]
    metricas = calcular_metricas(y_true, y_pred, y_proba)
    assert metricas == {"exactitud": 0.6667, "f1": 0.6667, "auc": 0.8889}
"""

In [ ]:
ruta_test_funciones = os.path.join(CARPETA, "test_funciones.py")
with open(ruta_test_funciones, "w", encoding="utf-8") as f:
    f.write(TEST_FUNCIONES_PY)
print("test unitari scritti in:", ruta_test_funciones)

In [ ]:
resumen = ejecutar_pytest([ruta_test_funciones])
print("test unitari superati:", resumen["passed"],
      "| falliti:", resumen["failed"],
      "| codice di uscita:", resumen["returncode"])
assert resumen["failed"] == 0 and resumen["returncode"] == 0, "i test unitari dovrebbero passare tutti"

**Cosa osservi?** Il riepilogo non mostra il testo grezzo di pytest (che include tempi come `"in 0,03s"`, diversi a ogni esecuzione): mostriamo solo i conteggi (superati/falliti) e il codice di uscita, estratti con un'espressione regolare. I 4 test passano: 2 per `limpiar_tickets_soporte`, 1 per `categorizar_antiguedad`, 1 per `calcular_metricas`.

---
## 3. Test dei dati con `pandera`

Un test dei dati non testa il codice: testa che **i dati stessi** rispettino delle regole (tipi, intervalli, valori ammessi, proporzione di nulli), qualunque sia il codice che li elabora dopo. `ESQUEMA_CHURN` (già scritto in `proyecto.py`, sezione 1) definisce, per ogni colonna di `churn.csv`, il suo tipo, i suoi valori validi e, a livello dell'intero `DataFrame`, che nessuna colonna abbia più del 5% di nulli.

Prima di formalizzare qualcosa in `pytest`, verifichiamo lo schema a mano: contro i dati reali (deve accettarli) e contro **due copie corrotte apposta**.

In [ ]:
sys.path.insert(0, CARPETA)
from proyecto import ESQUEMA_CHURN, cargar_datos, construir_pipeline, preparar_datos  # noqa: E402

df_real = pd.read_csv(cargar_datos("churn.csv"))
ESQUEMA_CHURN.validate(df_real)
print("lo schema accetta i dati reali: OK")

# Corrompiamo apposta una copia: antiguedad_meses negativa (impossibile)
df_antiguedad_mala = df_real.copy()
df_antiguedad_mala.loc[0, "antiguedad_meses"] = -5
try:
    ESQUEMA_CHURN.validate(df_antiguedad_mala)
    detectado_antiguedad = False
except pa.errors.SchemaError as error:
    detectado_antiguedad = True
    print("antiguedad_meses=-5 RIFIUTATA:", str(error).splitlines()[0][:90])
assert detectado_antiguedad, "lo schema dovrebbe rifiutare un'anzianità negativa"

# Corrompiamo apposta una copia: una categoria di plan inesistente
df_plan_malo = df_real.copy()
df_plan_malo.loc[0, "plan"] = "diamante"
try:
    ESQUEMA_CHURN.validate(df_plan_malo)
    detectado_plan = False
except pa.errors.SchemaError as error:
    detectado_plan = True
    print("plan='diamante' RIFIUTATO:", str(error).splitlines()[0][:90])
assert detectado_plan, "lo schema dovrebbe rifiutare una categoria di plan inesistente"

**Cosa osservi?** Lo schema accetta `churn.csv` così com'è, ma rifiuta immediatamente entrambe le corruzioni con un `pandera.errors.SchemaError` che dice esattamente quale colonna e quale regola è fallita. Un dato corrotto non arriva nemmeno alla `Pipeline` della lezione 5: si ferma qui, prima di addestrare o predire qualsiasi cosa.

In [ ]:
# Corrompiamo apposta una copia: 10% di nulli in factura_mensual (oltre il 5% permesso)
df_con_nulos = df_real.copy()
indices_nulos = df_con_nulos.sample(frac=0.10, random_state=SEMILLA).index
df_con_nulos.loc[indices_nulos, "factura_mensual"] = np.nan
proporcion_nulos = round(df_con_nulos["factura_mensual"].isna().mean(), 4)
print("proporzione di nulli introdotta in factura_mensual:", proporcion_nulos)

try:
    ESQUEMA_CHURN.validate(df_con_nulos)
    detectado_nulos = False
except pa.errors.SchemaError as error:
    detectado_nulos = True
    print("proporzione di nulli alta RIFIUTATA:", str(error).splitlines()[0][:90])
assert detectado_nulos, "lo schema dovrebbe rifiutare una proporzione di nulli oltre la soglia"

**Cosa osservi?** Con un 10% di nulli in `factura_mensual` (oltre il 5% permesso da `ESQUEMA_CHURN`), lo schema rifiuta anche questi dati: il controllo della proporzione di nulli è reale tanto quanto quello dei tipi o delle categorie.

### 3.1 Formalizziamo i tre casi in `pytest`

Ora scriviamo `proyecto_churn/test_datos.py`: un test che conferma che i dati reali passano, e tre test che usano `pytest.raises(pa.errors.SchemaError)` per confermare che ogni dato corrotto viene RIFIUTATO. Che un test "passi" qui significa che lo schema ha rilevato davvero il problema — se lo schema non lo rilevasse, `pytest.raises` fallirebbe e il test segnalerebbe l'errore.

In [ ]:
TEST_DATOS_PY = """
import numpy as np
import pandas as pd
import pandera.pandas as pa
import pytest
from proyecto import ESQUEMA_CHURN, cargar_datos

SEMILLA = 42


@pytest.fixture(scope="module")
def df_real():
    return pd.read_csv(cargar_datos("churn.csv"))


def test_esquema_valido_en_datos_reales(df_real):
    '''I dati reali del corso devono superare lo schema senza eccezioni.'''
    ESQUEMA_CHURN.validate(df_real)


def test_antiguedad_negativa_es_rechazada(df_real):
    '''Dato corrotto A: antiguedad_meses negativa (impossibile nella realtà).'''
    corrupto = df_real.copy()
    corrupto.loc[0, "antiguedad_meses"] = -5
    with pytest.raises(pa.errors.SchemaError):
        ESQUEMA_CHURN.validate(corrupto)


def test_categoria_de_plan_inexistente_es_rechazada(df_real):
    '''Dato corrotto B: una categoria di plan che non esiste nel catalogo.'''
    corrupto = df_real.copy()
    corrupto.loc[0, "plan"] = "diamante"
    with pytest.raises(pa.errors.SchemaError):
        ESQUEMA_CHURN.validate(corrupto)


def test_proporcion_alta_de_nulos_es_rechazada(df_real):
    '''Dato corrotto C: 10% di nulli in factura_mensual, oltre il 5% permesso.'''
    corrupto = df_real.copy()
    indices = corrupto.sample(frac=0.10, random_state=SEMILLA).index
    corrupto.loc[indices, "factura_mensual"] = np.nan
    with pytest.raises(pa.errors.SchemaError):
        ESQUEMA_CHURN.validate(corrupto)
"""

In [ ]:
ruta_test_datos = os.path.join(CARPETA, "test_datos.py")
with open(ruta_test_datos, "w", encoding="utf-8") as f:
    f.write(TEST_DATOS_PY)
print("test dei dati scritti in:", ruta_test_datos)

In [ ]:
resumen = ejecutar_pytest([ruta_test_datos])
print("test dei dati superati:", resumen["passed"],
      "| falliti:", resumen["failed"],
      "| codice di uscita:", resumen["returncode"])
assert resumen["passed"] == 4 and resumen["failed"] == 0, "i 4 test dei dati dovrebbero passare (1 valido + 3 casi corrotti rilevati)"

**Cosa osservi?** I 4 test dei dati passano (1 schema valido + 3 corruzioni rilevate). Nessuno dei tre `pytest.raises` è un test vuoto: se togliessi la riga `Check.isin([...])` di `plan` in `ESQUEMA_CHURN`, il test della categoria inesistente inizierebbe a FALLIRE davvero, perché lo schema non rileverebbe più il problema — così si sa che il test funziona e non passa "a caso".

---
## 4. Test del modello: una soglia di regressione

Un test del modello non testa il codice né i dati: testa che una **metrica del modello** non sia peggiorata più del tollerabile rispetto a una versione precedente. Definiamo `UMBRAL_F1 = F1_MODELO_ANTERIOR - TOLERANCIA`, con l'F1 precedente documentato in `datos/LEEME.md` (≈0,31, con `class_weight="balanced"`) e una tolleranza di 0,03.

In [ ]:
# F1 del modello precedente, documentato in datos/LEEME.md (config con class_weight="balanced")
F1_MODELO_ANTERIOR = 0.31
TOLERANCIA = 0.03
UMBRAL_F1 = round(F1_MODELO_ANTERIOR - TOLERANCIA, 4)
print("soglia di regressione (F1 minimo accettabile):", UMBRAL_F1)

### 4.1 Il caso che passa: il modello della lezione 5

In [ ]:
TEST_MODELO_PY = """
from sklearn.metrics import f1_score
from proyecto import construir_pipeline, preparar_datos

F1_MODELO_ANTERIOR = 0.31
TOLERANCIA = 0.03
UMBRAL_F1 = round(F1_MODELO_ANTERIOR - TOLERANCIA, 4)


def test_modelo_actual_no_empeora_mas_de_la_tolerancia():
    '''Test di REGRESSIONE: il modello nuovo non può essere peggiore del precedente di più di TOLERANCIA.'''
    X_train, X_test, y_train, y_test = preparar_datos()
    pipeline = construir_pipeline(balanceado=True)
    pipeline.fit(X_train, y_train)
    f1_actual = round(f1_score(y_test, pipeline.predict(X_test)), 4)
    assert f1_actual >= UMBRAL_F1, f"F1 attuale {f1_actual} sotto la soglia {UMBRAL_F1}"
"""

In [ ]:
ruta_test_modelo = os.path.join(CARPETA, "test_modelo.py")
with open(ruta_test_modelo, "w", encoding="utf-8") as f:
    f.write(TEST_MODELO_PY)
print("test del modello scritto in:", ruta_test_modelo)

In [ ]:
resumen = ejecutar_pytest([ruta_test_modelo])
print("test del modello (caso che SUPERA la soglia) superati:", resumen["passed"],
      "| falliti:", resumen["failed"],
      "| codice di uscita:", resumen["returncode"])
assert resumen["passed"] == 1 and resumen["failed"] == 0, "il modello della lezione 5 dovrebbe superare la soglia"

**Cosa osservi?** L'F1 della `Pipeline` bilanciata (0,3166, la stessa cifra della lezione 5) supera la soglia (0,28): il test passa, e con un unico comando (`pytest`) resta traccia automatica che il modello attuale non è peggiorato rispetto al precedente.

### 4.2 Il caso che fallisce apposta: un modello peggiore, costruito senza bilanciamento

In [ ]:
TEST_MODELO_PEOR_PY = """
from sklearn.metrics import f1_score
from proyecto import construir_pipeline, preparar_datos

F1_MODELO_ANTERIOR = 0.31
TOLERANCIA = 0.03
UMBRAL_F1 = round(F1_MODELO_ANTERIOR - TOLERANCIA, 4)


def test_modelo_sin_balancear_supera_el_umbral():
    '''Modello peggiore COSTRUITO APPOSTA: senza class_weight="balanced" (demo che il test FALLISCE davvero).'''
    X_train, X_test, y_train, y_test = preparar_datos()
    pipeline = construir_pipeline(balanceado=False)
    pipeline.fit(X_train, y_train)
    f1_actual = round(f1_score(y_test, pipeline.predict(X_test)), 4)
    assert f1_actual >= UMBRAL_F1, f"F1 attuale {f1_actual} sotto la soglia {UMBRAL_F1}"
"""

In [ ]:
ruta_test_peor = os.path.join(CARPETA, "test_modelo_peor_demo.py")
with open(ruta_test_peor, "w", encoding="utf-8") as f:
    f.write(TEST_MODELO_PEOR_PY)

# Eseguiamo SOLO questo file, a parte dalla suite, per dimostrare che il test rileva un modello peggiore
resumen_peor = ejecutar_pytest([ruta_test_peor])
print("demo (modello senza bilanciamento) superati:", resumen_peor["passed"],
      "| falliti:", resumen_peor["failed"],
      "| codice di uscita:", resumen_peor["returncode"])
assert resumen_peor["failed"] == 1 and resumen_peor["returncode"] != 0, "il test dovrebbe FALLIRE davvero con un modello peggiore: se non fallisce, il test non serve a niente"

# Eliminiamo il file dimostrativo: NON fa parte della suite finale
os.remove(ruta_test_peor)
print("file dimostrativo eliminato (non appartiene alla suite finale).")

**Cosa osservi?** Senza `class_weight="balanced"`, il modello predice quasi sempre "non abbandona" (la classe maggioritaria): il suo F1 reale è molto basso, ben sotto la soglia 0,28, e il test FALLISCE davvero (1 fallimento, codice di uscita diverso da 0). Questa è la prova che il test di regressione funziona: non passa sempre, passa solo quando il modello rispetta davvero la soglia. Eliminiamo il file dimostrativo perché non deve far parte della suite finale (sezione 5).

---
### 4.3 I limiti di una soglia fissa

Una soglia di regressione è utile, ma ha limiti reali. Due esempi illustrativi, con numeri.

**(a) Non rileva una deriva lenta e progressiva.** Se il test confronta ogni versione solo con quella IMMEDIATAMENTE precedente, un calo piccolo ogni volta (minore della tolleranza) non fa mai scattare il test, anche se il calo accumulato è grave.

In [ ]:
# Esempio illustrativo: F1 misurato ogni settimana, in calo poco a poco
semanas = list(range(11))
f1_semanal = [round(0.3166 - 0.005 * s, 4) for s in semanas]
print("F1 settimana per settimana:", f1_semanal)

caidas_semanales = [round(f1_semanal[i - 1] - f1_semanal[i], 4) for i in range(1, len(f1_semanal))]
pasa_semana_a_semana = all(caida <= TOLERANCIA for caida in caidas_semanales)
print("il test settimana-contro-settimana-precedente passa TUTTE le settimane?", pasa_semana_a_semana)

primera_semana_bajo_umbral = next(s for s, f in zip(semanas, f1_semanal) if f < UMBRAL_F1)
print("prima settimana con F1 sotto la soglia ORIGINALE (0,28):", primera_semana_bajo_umbral, f1_semanal[primera_semana_bajo_umbral])

assert pasa_semana_a_semana, "ogni calo settimanale è minore della tolleranza: per costruzione di questo esempio"
assert f1_semanal[primera_semana_bajo_umbral] < UMBRAL_F1

**Cosa osservi?** (*Esempio illustrativo*, con un calo costante di 0,005 a settimana.) Ogni confronto settimana-contro-settimana-precedente passa (0,005 < tolleranza 0,03), quindi un test che guardasse solo "sono peggiorato rispetto alla settimana scorsa?" non scatterebbe mai. Ma confrontato con la soglia ORIGINALE (0,28), l'F1 è già sotto nella settimana 8 (0,2766) — 8 settimane di silenzio prima che qualcosa si noti, se nessuno confronta con il punto di partenza.

**(b) Non rileva un problema in un sottogruppo specifico, anche se la media regge.** Riusiamo il modello buono della sezione 4.1 e guardiamo l'F1 separato per `plan`.

In [ ]:
# Riusiamo il modello buono (bilanciato) della sezione 4 e guardiamo l'F1 PER PLAN
X_train, X_test, y_train, y_test = preparar_datos()
pipeline_bueno = construir_pipeline(balanceado=True)
pipeline_bueno.fit(X_train, y_train)
predicciones = pipeline_bueno.predict(X_test)

f1_global = round(f1_score(y_test, predicciones), 4)
f1_por_plan = {}
for plan in sorted(X_test["plan"].unique()):
    filtro = (X_test["plan"] == plan).values
    f1_por_plan[plan] = round(f1_score(y_test[filtro], predicciones[filtro]), 4)

print("F1 globale (intero set di test):", f1_global)
print("F1 per plan:", f1_por_plan)
assert f1_por_plan["premium"] < f1_global, "il sottogruppo premium dovrebbe rendere peggio della media globale"

**Cosa osservi?** L'F1 globale (0,3166) supera comodamente la soglia (0,28). Ma per plan, l'F1 varia parecchio: il sottogruppo `premium` rende peggio della media globale, mentre `basico` e `estandar` la sostengono. Un test che guardi solo l'F1 globale non vedrebbe nessun problema — il sottogruppo con il rendimento peggiore resta nascosto dietro la media, esattamente il limite descritto in questa sezione.

> **Attenzione:** nessuna soglia fissa, per quanto ben scelta, sostituisce la sorveglianza continua (media mobile, confronto con il punto di partenza, metriche per sottogruppo). Una soglia di regressione è una RETE DI SICUREZZA minima prima di pubblicare una nuova versione, non un sistema di monitoraggio completo — questo arriva nelle lezioni successive del corso.

---
## 5. Un'unica suite: i tre tipi di test con un solo comando

`proyecto_churn/` contiene ora `test_funciones.py` (unitari), `test_datos.py` (dati) e `test_modelo.py` (modello) — il file dimostrativo del modello peggiore è già stato eliminato. Un unico comando, `pytest` sulla cartella, esegue insieme i tre tipi di test.

In [ ]:
resumen_total = ejecutar_pytest([CARPETA])
print("SUITE COMPLETA (unitari + dati + modello) — superati:", resumen_total["passed"],
      "| falliti:", resumen_total["failed"],
      "| codice di uscita:", resumen_total["returncode"])
assert resumen_total["returncode"] == 0 and resumen_total["failed"] == 0, "la suite completa dovrebbe passare con un unico comando"
assert resumen_total["passed"] == 9, "4 unitari + 4 dei dati + 1 del modello = 9"

**Cosa osservi?** 9 test superati (4 unitari + 4 dei dati + 1 del modello), 0 falliti, codice di uscita 0 — esattamente ciò che una CI/CD (lezione 8) eseguirebbe prima di accettare una modifica: un unico comando che copre codice, dati e modello insieme.

---
## 6. Pulizia finale

Eliminiamo `proyecto_churn/` (con `proyecto.py` e i tre file di test) perché non resti nessun residuo dentro il repository del corso.

In [ ]:
shutil.rmtree(CARPETA)
print("Cartella di lavoro eliminata:", CARPETA)
assert not os.path.exists(CARPETA)

---
## Riepilogo della lezione

- I **test unitari** (`pytest`) testano funzioni reali di pulizia, trasformazione e valutazione, con casi normali e casi limite.
- I **test dei dati** (`pandera`) testano che i dati stessi rispettino uno schema (tipi, intervalli, categorie, proporzione di nulli), con corruzioni apposta che dimostrano che lo schema rileva davvero i problemi.
- Il **test del modello** confronta una metrica attuale con una soglia di regressione (F1 precedente meno una tolleranza): passa con il modello buono della lezione 5 e FALLISCE davvero con un modello peggiore costruito apposta (senza bilanciamento) — la prova che il test funziona.
- Una soglia fissa ha dei limiti: non rileva una deriva lenta (ogni passo dentro la tolleranza, ma il calo accumulato no) né un problema localizzato in un sottogruppo (qui, il plan `premium`) finché la media regge.
- I tre tipi di test convivono nella stessa cartella e si eseguono insieme con un unico comando: `pytest`.

*Continua con la lezione 7: Servire un modello.*